# 04d -- Stage 4d: Add AI-generated video beyond face-swap

The custom model's only video training data so far is DFDC -- face-swap
deepfakes specifically. It has never seen AI video from a modern
text-to-video generator (no face-swap involved at all, often no face
present), which is now a meaningfully different artifact profile.

**[GenBuster-200K-mini](https://huggingface.co/datasets/l8cv/GenBuster-200K-mini)**
(MIT, **not gated** -- correcting this project's own earlier planning
doc, which assumed it required a terms-acceptance click; the live HF API
confirms open access, no token needed) is a single 5.46GB zip, a small
subset of the full 200K-clip/114GB GenBuster-200K (100K real + 100K
AI-generated, built for the BusterX paper). The full dataset ships as 27
split 7z archives -- the mini version is the only practical size for a
free Kaggle/Colab GPU quota.

**VidProM** (1.67M prompts / 6.69M videos across 4 generators) is
documented here as a stretch option, not implemented: it's 3.19TB total,
and its license is mixed -- 2 of its 4 generator subsets (Pika,
ModelScope) are CC BY-NC 4.0 (non-commercial), the other 2
(VideoCraft2: Apache-2.0, Text2Video-Zero: CreativeML Open RAIL-M) have
different terms again. Its own `example/` folder (10,000 datapoints,
pulled via a path-filtered `hf_hub_download` rather than the full
snapshot) would be the practical entry point if this gets revisited.

**This notebook's internal structure is genuinely unconfirmed** --
GenBuster-200K-mini's own README is essentially empty, and nothing
public documents its internal folder/filename layout. Unlike Stage 4c
(where the 10-style x 3-source structure was cross-confirmed by
multiple independent sources even though exact folder casing wasn't),
here there is **no prior confirmation of any kind** -- the verify-first
cell below is not a formality, it's the only way to know what's
actually in this zip. Read its output carefully and adjust the
classifier cell if what it finds doesn't match the keyword-matching
logic's assumptions.

Follows Stage 2's frame-extraction pattern (sample frames at a target
FPS, cap frames/clip, resize to the model's input size) -- but skips
Stage 2's face-cropping step, since this is general AI-generated video
content, not face-swap; there's no reason to assume a face is even
present in every clip.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import os, zipfile
from pathlib import Path
import pandas as pd
import cv2
from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.video.extractor import extract_frames, get_video_fps, get_frame_count
from services.video.sampler import sample_frame_indices

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
print("Dataset root:", dataset_root)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".webm", ".mkv"}


def find_videos(root):
    """os.walk(followlinks=True), not Path.glob -- same fix as Stage 2's
    DFDC extraction and Stage 4c: pathlib's recurse_symlinks doesn't
    reliably follow Drive/Kaggle-mounted symlinks, plain os.walk does."""
    found = []
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        for fn in filenames:
            if Path(fn).suffix.lower() in VIDEO_EXTS:
                found.append(Path(dirpath) / fn)
    return found

## Download GenBuster-200K-mini from Hugging Face

Not gated -- no HF token needed, `hf_hub_download` works directly.
Idempotent: skips the download if the zip is already present locally.

In [ ]:
local_root = Path("/content/extra_datasets") if IS_COLAB else Path("/tmp/extra_datasets")
local_root.mkdir(parents=True, exist_ok=True)
genbuster_dir = local_root / "genbuster_mini"

from huggingface_hub import hf_hub_download

zip_path = local_root / "GenBuster-200K-mini.zip"
if not zip_path.exists():
    downloaded = hf_hub_download(
        repo_id="l8cv/GenBuster-200K-mini",
        repo_type="dataset",
        filename="GenBuster-200K-mini.zip",
        local_dir=str(local_root),
    )
    print(f"Downloaded -> {downloaded}")
else:
    print(f"{zip_path} already downloaded.")

marker = genbuster_dir / ".extracted"
if not marker.exists():
    genbuster_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {zip_path} -> {genbuster_dir} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(genbuster_dir)
    marker.touch()
    print("Done.")
else:
    print(f"{genbuster_dir} already extracted, skipping.")

## Verify the actual structure before trusting anything

**Stop and actually read this output before running anything past it.**
Prints every directory name and a sample of filenames found anywhere in
the extracted tree -- there is zero prior confirmation of this
dataset's internal layout (unlike Stage 4c, where at least the overall
structure was cross-confirmed), so this is the only source of truth
available. If real/fake isn't obviously signaled by a folder name or a
filename keyword in what gets printed, the classifier cell below needs
to be rewritten to match whatever the real pattern turns out to be --
don't assume the default keyword guesses below are correct.

In [ ]:
if genbuster_dir.exists():
    all_dirnames = sorted({p.name for p in genbuster_dir.rglob("*") if p.is_dir()})
    print(f"{len(all_dirnames)} unique directory names:")
    for name in all_dirnames:
        print(" ", name)

    sample_videos = find_videos(genbuster_dir)[:20]
    print(f"\n{len(find_videos(genbuster_dir))} video files total. Sample filenames:")
    for p in sample_videos:
        print(" ", p.relative_to(genbuster_dir))
else:
    print(f"{genbuster_dir} doesn't exist yet -- nothing to verify.")

## Classify real vs. AI-generated

Classifies by the **immediate parent folder name and the filename
itself only** -- not any folder further up the tree. A first version
also checked the grandparent folder as a fallback and used a bare
"gen"/"ai" substring match; testing caught that this incorrectly
flagged files as fake purely because the dataset's own extracted root
folder is named `GenBuster-200K-mini`, which itself contains "gen" --
exactly the same class of bug Stage 4c's outer-folder-name contamination
was, just one keyword further up. Restricting to the closest folder only,
and requiring more specific keywords ("generated", "synthetic", "gan",
"ai_"/"_ai" rather than bare "ai"/"gen"), avoids it.

**Rewrite this function first if the verify-structure cell above showed
something this doesn't account for** -- e.g. if clips are split by
generator-name subfolders ("sora/", "runway/", etc.) with no explicit
real/fake keyword at all, this intentionally returns `None` (unmatched)
rather than guess, since a wrong silent guess is worse than a visible
gap you can go fix.

`MAX_PER_CLASS` caps clips sampled per class, same reasoning as Stage
4c: this is meant to add general AI-video coverage alongside DFDC, not
dwarf every other dataset in the merged manifest.

In [ ]:
import random

MAX_PER_CLASS = 1500  # clips per real/fake class, not per generator -- adjust as needed

def classify_real_fake(path):
    # Immediate parent folder + filename only -- NOT any folder further
    # up the tree. Confirmed by testing: the extracted root folder name
    # itself (GenBuster-200K-mini) contains "gen", which a grandparent
    # fallback + bare "gen"/"ai" keywords would have matched on anything
    # whose immediate parent didn't otherwise classify.
    candidates = [path.parent.name.lower(), path.stem.lower()]
    for text in candidates:
        if "real" in text:
            return "real"
        if any(kw in text for kw in ("fake", "generated", "synthetic", "ai_", "_ai", "gan")):
            return "fake"
    return None


genbuster_candidates = {"real": [], "fake": []}
unmatched = 0
if genbuster_dir.exists():
    for p in find_videos(genbuster_dir):
        label = classify_real_fake(p)
        if label is None:
            unmatched += 1
            continue
        genbuster_candidates[label].append(p)

print(f"Matched: {len(genbuster_candidates['real'])} real, "
      f"{len(genbuster_candidates['fake'])} fake, {unmatched} unmatched")

random.seed(42)
genbuster_clips = []
for label, paths in genbuster_candidates.items():
    sampled = random.sample(paths, min(len(paths), MAX_PER_CLASS))
    genbuster_clips.extend((p, label) for p in sampled)
    print(f"  sampled {len(sampled)}/{len(paths)} {label} clips (cap={MAX_PER_CLASS})")

## Extract frames from matched clips

Same sampling approach as Stage 2's DFDC extraction (target FPS from
`settings.video.sample_fps`, capped at `FRAMES_PER_VIDEO` per clip,
resized to the model's input size before saving -- saving at full video
resolution only wastes disk with no training-time benefit, since
`preprocess_image()` resizes to this anyway). **No face-cropping** --
that's specific to DFDC's face-swap content, not applicable here.

Resumable: writes a progress file and appends to the frame manifest as
it goes, so interrupting and rerunning this cell doesn't restart from
scratch or duplicate already-extracted clips.

In [ ]:
FRAMES_PER_VIDEO = 5

frames_out_dir = resolve_path("data/synthetic_data")
frames_out_dir.mkdir(parents=True, exist_ok=True)
frame_manifest_path = resolve_path("data/genbuster_frame_manifest.csv")
progress_path = resolve_path("data/genbuster_processed_clips.txt")

processed_clips = set(progress_path.read_text().splitlines()) if progress_path.exists() else set()
total_rows_written = sum(1 for _ in open(frame_manifest_path)) - 1 if frame_manifest_path.exists() else 0
if processed_clips:
    print(f"Resuming: {total_rows_written} rows already written, {len(processed_clips)} clip(s) already done.")

for video_path, label in genbuster_clips:
    clip_key = str(video_path)
    if clip_key in processed_clips:
        continue

    manip_type = "none" if label == "real" else "gan"
    fps = get_video_fps(str(video_path))
    total = get_frame_count(str(video_path))
    indices = set(sample_frame_indices(total, fps, settings.video.sample_fps, max_frames=FRAMES_PER_VIDEO))

    rows = []
    for idx, frame in extract_frames(str(video_path)):
        if idx not in indices:
            continue
        frame_name = f"genbuster_{video_path.stem}_frame{idx}.jpg"
        frame = cv2.resize(frame, tuple(settings.inference.image_size))
        cv2.imwrite(str(frames_out_dir / frame_name), frame)
        rows.append({"filename": str(frames_out_dir / frame_name), "label": label, "type": manip_type})

    if rows:
        pd.DataFrame(rows, columns=["filename", "label", "type"]).to_csv(
            frame_manifest_path, mode="a", header=(total_rows_written == 0), index=False
        )
        total_rows_written += len(rows)

    with open(progress_path, "a") as f:
        f.write(clip_key + "\n")

print(f"Done. {total_rows_written} total frame rows written.")

## Bundle extracted frames into one zip for Drive

Same reasoning as every other large-row-count source in this project:
writing thousands of individual small files straight onto Drive's FUSE
mount is the slow/fragile trap already hit more than once. One zip is a
single fast write instead.

In [ ]:
if total_rows_written > 0 and IS_COLAB:
    import shutil

    bundle_path = local_root / "genbuster_frames_bundle.zip"
    manifest_df = pd.read_csv(frame_manifest_path)
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _, row in manifest_df.iterrows():
            src = Path(row["filename"])
            if src.exists():
                zf.write(src, src.name)
        zf.writestr("genbuster_manifest.csv", manifest_df.to_csv(index=False))
    shutil.copy(bundle_path, dataset_root / bundle_path.name)
    print(f"Bundled -> Drive: {dataset_root / bundle_path.name} "
          f"({bundle_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("Nothing to bundle, or not on Colab -- skipping.")

## Merge into the training manifest

**Check the printed counts above first.** Folds the extracted-frame rows
into Stage 3's `merged_labels.csv`, same shared manifest every other
stage builds onto. Re-running this notebook replaces its previous rows
instead of duplicating them.

In [ ]:
merged_path = dataset_root / "merged_labels.csv"
base_df = pd.read_csv(merged_path) if merged_path.exists() else pd.DataFrame(columns=["filename", "label", "type"])

extra_df = pd.read_csv(frame_manifest_path) if frame_manifest_path.exists() else pd.DataFrame(columns=["filename", "label", "type"])


def _is_ours(filename):
    return "genbuster_" in Path(filename).name


base_df = base_df[~base_df["filename"].astype(str).apply(_is_ours)]
merged_df = pd.concat([base_df, extra_df], ignore_index=True)
merged_df.to_csv(merged_path, index=False)
print(f"merged_labels.csv now has {len(merged_df)} rows ({len(extra_df)} from GenBuster-200K-mini this run).")

## Next step

Rerun Stage 3's retrain cells (now reading the updated
`merged_labels.csv`) from the latest checkpoint -- same training loop,
no path to change by hand.